In [1]:
import pandas as pd

python_pr = pd.read_parquet("python_pr.parquet")

In [ ]:
python_pr

# github token
# 

,html_url,error,files,owner,repo,pr_number,merged,merged_at,merge_commit_sha,hit_exts,repo_url
0,https://github.com/mlflow/mlflow/pull/16449,None,"[{'additions': 6, 'changes': 8, 'deletions': 2...",mlflow,mlflow,16449.0,1.0,2025-06-25 12:35:55+00:00,a05c8bfeb79593d5afe8951379e0396131045028,[.py],https://api.github.com/repos/mlflow/mlflow
1,https://github.com/MervinPraison/PraisonAI/pul...,None,"[{'additions': 231, 'changes': 231, 'deletions...",MervinPraison,PraisonAI,1021.0,1.0,2025-07-19 08:45:51+00:00,893026b620186298e6f7ea96b25d872c3e226c83,[.py],https://api.github.com/repos/MervinPraison/Pra...
2,https://github.com/typedef-ai/fenic/pull/81,None,"[{'additions': 6, 'changes': 7, 'deletions': 1...",typedef-ai,fenic,81.0,1.0,2025-07-14 18:58:49+00:00,c498741e5ab25c3e15af16dc7c5499b85ed31ec2,[.py],https://api.github.com/repos/typedef-ai/fenic
3,https://github.com/BeehiveInnovations/zen-mcp-...,None,"[{'additions': 2, 'changes': 4, 'deletions': 2...",BeehiveInnovations,zen-mcp-server,95.0,1.0,2025-06-20 20:08:11+00:00,69a3121452738f0e35634521d08fb10caa870663,[.py],https://api.github.com/repos/BeehiveInnovation...
4,https://github.com/INCATools/ontology-access-k...,None,"[{'additions': 2, 'changes': 2, 'deletions': 0...",INCATools,ontology-access-kit,831.0,1.0,2025-03-11 22:21:15+00:00,c0e053c146937969e9b8e6358f731c6eb87408ef,[.py],https://api.github.com/repos/INCATools/ontolog...
...,...,...,...,...,...,...,...,...,...,...,...
3373,https://github.com/MontrealAI/AGI-Alpha-Agent-...,None,"[{'additions': 2, 'changes': 2, 'deletions': 0...",MontrealAI,AGI-Alpha-Agent-v0,3657.0,1.0,2025-07-22 13:24:06+00:00,b9b474a9057efaecbcbf868f55285bb4b81c0fea,[.py],https://api.github.com/repos/MontrealAI/AGI-Al...
3374,https://github.com/MontrealAI/AGI-Alpha-Agent-...,None,"[{'additions': 1, 'changes': 1, 'deletions': 0...",MontrealAI,AGI-Alpha-Agent-v0,1173.0,1.0,2025-05-30 00:39:51+00:00,46d0edd86719d9c5691931dacfc274245d9818d9,[.py],https://api.github.com/repos/MontrealAI/AGI-Al...
3375,https://github.com/commaai/panda/pull/2227,None,"[{'additions': 0, 'changes': 130, 'deletions':...",commaai,panda,2227.0,1.0,2025-07-19 16:55:13+00:00,3d27233530178660e593535efa63681819d292fc,[.py],https://api.github.com/repos/commaai/panda
3376,https://github.com/sooperset/mcp-atlassian/pul...,None,"[{'additions': 9, 'changes': 15, 'deletions': ...",sooperset,mcp-atlassian,540.0,1.0,2025-06-23 10:22:20+00:00,09d9ae860c0264d25bae966fe8485e2b13f8eff7,[.py],https://api.github.com/repos/sooperset/mcp-atl...


In [ ]:
import re
import requests

GITHUB_TOKEN = " "

def parse_pr_url(pr_url: str):
    m = re.match(r"https://github\.com/([^/]+)/([^/]+)/pull/(\d+)", pr_url)
    if not m:
        raise ValueError(f"Invalid PR URL: {pr_url}")
    owner, repo, pr_number = m.group(1), m.group(2), int(m.group(3))
    return owner, repo, pr_number

def fetch_pr_metadata(pr_url: str):
    owner, repo, pr_number = parse_pr_url(pr_url)

    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}"
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {GITHUB_TOKEN}",
    }

    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json()

    return {
        "owner": owner,
        "repo": repo,
        "pr_number": pr_number,
        "base_ref": data["base"]["ref"],
        "base_sha": data["base"]["sha"],
        "head_ref": data["head"]["ref"],
        "head_sha": data["head"]["sha"],
        "merge_commit_sha": data.get("merge_commit_sha"),
        "merged": data.get("merged"),
    }


import requests

headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
}

def fetch_base_sha(owner, repo, pr_number):
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}"

    r = requests.get(url, headers=headers, timeout=30)

    if r.status_code != 200:
        return None

    data = r.json()
    return data["base"]["sha"]



In [5]:
from tqdm import tqdm
import time

base_shas = []

for url in tqdm(python_pr["html_url"]):

    parsed = parse_pr_url(url)

    if not parsed:
        base_shas.append(None)
        continue

    owner, repo, pr_number = parsed

    try:
        base_sha = fetch_base_sha(owner, repo, pr_number)
    except Exception:
        base_sha = None

    base_shas.append(base_sha)

    time.sleep(0.1)

python_pr["base_sha"] = base_shas

100%|██████████| 3378/3378 [46:10<00:00,  1.22it/s]  


In [12]:
python_pr.groupby(["owner","repo","base_sha"]).size().sort_values(ascending=False).head(20)

owner          repo                  base_sha                                
Ljzd-PRO       KToolBox              e6b8176a5f89ec4b20367271413874c33839d9e1    6
econ-ark       HARK                  fc815b59080de7620ffb0e289f19c318555d1274    5
peass-ng       PEASS-ng              c3a93a57fe91996d4d4f48a6201365b22df00baa    4
goat-sdk       goat                  1bcd45cf7d4adc7bf99b84b57670c4f25dfb673b    4
MontrealAI     AGI-Alpha-Agent-v0    930a0f1783452e8ae264422e68614570b01c5f72    4
PriorLabs      TabPFN                6102d8b175c5e66c4790469be2b8d9ec034c6a68    4
MontrealAI     AGI-Alpha-Agent-v0    20144dd95b7bf54c3bb75357f1f174e8db7c1b6b    4
tomquist       b2500-meter           e5eb53925a3bbc2d484ba09618198c944398b17d    3
joewandy       hlda                  34f35ccd58b0e0f2582b14ae9812d8294fadf7d2    3
marimo-team    marimo                fa5ed75bfd6e24e4d2e7fb9328cfadf0ac370b71    3
MontrealAI     AGI-Alpha-Agent-v0    571c23c83c6b691c9df57bd17d116347ef5ca147    3
joewandy 

In [9]:
python_pr.to_parquet("python_pr_with_base_sha_merge_commit_sha.parquet")

In [13]:
python_pr.head()

,html_url,error,files,owner,repo,pr_number,merged,merged_at,merge_commit_sha,hit_exts,repo_url,base_sha
0,https://github.com/mlflow/mlflow/pull/16449,None,"[{'additions': 6, 'changes': 8, 'deletions': 2...",mlflow,mlflow,16449.0,1.0,2025-06-25 12:35:55+00:00,a05c8bfeb79593d5afe8951379e0396131045028,[.py],https://api.github.com/repos/mlflow/mlflow,39a26815c059e352c6bc09fa23d1f5d3a99e545d
1,https://github.com/MervinPraison/PraisonAI/pul...,None,"[{'additions': 231, 'changes': 231, 'deletions...",MervinPraison,PraisonAI,1021.0,1.0,2025-07-19 08:45:51+00:00,893026b620186298e6f7ea96b25d872c3e226c83,[.py],https://api.github.com/repos/MervinPraison/Pra...,52d2a565ed1e51a524ad468f30780410fc5fbcb2
2,https://github.com/typedef-ai/fenic/pull/81,None,"[{'additions': 6, 'changes': 7, 'deletions': 1...",typedef-ai,fenic,81.0,1.0,2025-07-14 18:58:49+00:00,c498741e5ab25c3e15af16dc7c5499b85ed31ec2,[.py],https://api.github.com/repos/typedef-ai/fenic,acb0baa37f1b7fd29a1b970bab754a21634e309c
3,https://github.com/BeehiveInnovations/zen-mcp-...,None,"[{'additions': 2, 'changes': 4, 'deletions': 2...",BeehiveInnovations,zen-mcp-server,95.0,1.0,2025-06-20 20:08:11+00:00,69a3121452738f0e35634521d08fb10caa870663,[.py],https://api.github.com/repos/BeehiveInnovation...,4dae6e457ee05fb1042cd853d450c203e7985672
4,https://github.com/INCATools/ontology-access-k...,None,"[{'additions': 2, 'changes': 2, 'deletions': 0...",INCATools,ontology-access-kit,831.0,1.0,2025-03-11 22:21:15+00:00,c0e053c146937969e9b8e6358f731c6eb87408ef,[.py],https://api.github.com/repos/INCATools/ontolog...,59937f7136e3b6d04668ac3e50dfbb5448f0dfe0


In [17]:
sample = python_pr.iloc[1]
sample


html_url            https://github.com/MervinPraison/PraisonAI/pul...
error                                                            None
files               [{'additions': 231, 'changes': 231, 'deletions...
owner                                                   MervinPraison
repo                                                        PraisonAI
pr_number                                                      1021.0
merged                                                            1.0
merged_at                                   2025-07-19 08:45:51+00:00
merge_commit_sha             893026b620186298e6f7ea96b25d872c3e226c83
hit_exts                                                        [.py]
repo_url            https://api.github.com/repos/MervinPraison/Pra...
base_sha                     52d2a565ed1e51a524ad468f30780410fc5fbcb2
Name: 1, dtype: object